# CLUSE-Test: Fixed-Cluster High-Effort EvoEval Runner

This notebook runs CLUSE-Test on the **500 semantic-altering EvoEval tasks**. The mutant partition is created once before Layer 1 and reused unchanged in Layers 2 and 3; later layers operate only on surviving members of those original clusters. OpenAI calls use **high reasoning effort**, and the project does not impose a fixed output-token ceiling.

Run the mock smoke test first. Mock outputs validate integration only and must not be reported as research results.


## 1. Copy the attached project into Kaggle working storage

Kaggle mounts Dataset inputs as read-only folders. This cell automatically finds the attached CLUSE-Test project containing `run_pipeline.py` and copies it to `/kaggle/working`.

In [ ]:
import os
from pathlib import Path

# The 500-task EvoEval Parquet file is already built (see data/EVOEVAL_DATASET.md).
# This project no longer builds it from the source EvoEval Hugging Face repositories --
# point CLUSE_DATASET at wherever you uploaded/attached the file, or place it at the
# default path below (e.g. via a Kaggle Dataset attached with Add Input).
EVOEVAL_DIR = Path(os.environ.get('CLUSE_DATASET_DIR', '/kaggle/input/evoeval-semantic-500'))
DATASET_PATH = Path(os.environ.get('CLUSE_DATASET', str(EVOEVAL_DIR / 'EvoEval_semantic_500.parquet')))
DATASET_MANIFEST = EVOEVAL_DIR / 'EvoEval_semantic_500_manifest.json'

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found at {DATASET_PATH}. Attach the pre-built EvoEval Parquet '
        'file as a Kaggle Dataset input, or set CLUSE_DATASET to its path.'
    )

print('Using existing dataset:', DATASET_PATH)
print('Parquet exists:', DATASET_PATH.exists())
print('Manifest exists:', DATASET_MANIFEST.exists())
print('Parquet size (MB):', round(DATASET_PATH.stat().st_size / (1024 ** 2), 2))


## 2. Install dependencies

In [ ]:
%pip install -q -r requirements.txt

## 3. Load API secrets safely

Create a Kaggle Secret named `OPENAI_API_KEY` and enable it for this notebook. `HF_TOKEN` is optional because the EvoEval repositories are public, but it can improve Hugging Face reliability.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception as exc:
    secrets = None
    print('Kaggle Secrets client unavailable:', type(exc).__name__)

for secret_name in ('OPENAI_API_KEY', 'HF_TOKEN'):
    if secrets is None:
        continue
    try:
        value = secrets.get_secret(secret_name)
    except Exception:
        value = None
    if value:
        os.environ[secret_name] = value
        print(f'{secret_name} loaded.')
    else:
        print(f'{secret_name} not configured.')

print('OpenAI key available:', bool(os.environ.get('OPENAI_API_KEY')))
print('HF token available:', bool(os.environ.get('HF_TOKEN')))

## 4. Build the finalized EvoEval dataset

The builder downloads the five official semantic-altering subsets, gives every task a unique subset-qualified ID, preserves the original EvoEval ID, infers its HumanEval parent, validates the 100 × 5 distribution, and writes a Parquet file plus a SHA-256 manifest.

Turn Kaggle Internet **On** before running this cell.

In [ ]:
import os
from pathlib import Path

# The 500-task EvoEval Parquet file is already built (see data/EVOEVAL_DATASET.md).
# This project no longer builds it from the source EvoEval Hugging Face repositories --
# point CLUSE_DATASET at wherever you uploaded/attached the file, or place it at the
# default path below (e.g. via a Kaggle Dataset attached with Add Input).
EVOEVAL_DIR = Path(os.environ.get('CLUSE_DATASET_DIR', '/kaggle/input/evoeval-semantic-500'))
DATASET_PATH = Path(os.environ.get('CLUSE_DATASET', str(EVOEVAL_DIR / 'EvoEval_semantic_500.parquet')))
DATASET_MANIFEST = EVOEVAL_DIR / 'EvoEval_semantic_500_manifest.json'

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found at {DATASET_PATH}. Attach the pre-built EvoEval Parquet '
        'file as a Kaggle Dataset input, or set CLUSE_DATASET to its path.'
    )

print('Using existing dataset:', DATASET_PATH)
print('Parquet exists:', DATASET_PATH.exists())
print('Manifest exists:', DATASET_MANIFEST.exists())
print('Parquet size (MB):', round(DATASET_PATH.stat().st_size / (1024 ** 2), 2))


## 5. Validate the finalized dataset

In [ ]:
import json
import pandas as pd
from IPython.display import display

manifest = json.loads(DATASET_MANIFEST.read_text(encoding='utf-8'))
print(json.dumps(manifest, indent=2))

metadata_df = pd.read_parquet(
    DATASET_PATH,
    columns=['task_id', 'source_task_id', 'parent_task_id', 'evoeval_subset', 'entry_point'],
)

print()
print('Rows:', len(metadata_df))
print('Unique task IDs:', metadata_df['task_id'].nunique())
print('Unique HumanEval parents:', metadata_df['parent_task_id'].nunique())
print()
print('Subset distribution:')
display(metadata_df.groupby('evoeval_subset').size().rename('tasks').reset_index())

assert len(metadata_df) == 500
assert metadata_df['task_id'].nunique() == 500
assert set(metadata_df.groupby('evoeval_subset').size()) == {100}
assert metadata_df['parent_task_id'].notna().all()
print('Dataset validation passed.')

## 6. Inspect normalized CLUSE-Test tasks

In [ ]:
from src.utils.dataset_loader import load_dataset, select_problems

DATASET_TYPE = 'evoeval'
problems = load_dataset(DATASET_PATH, dataset_type=DATASET_TYPE)
first = problems[0]

print('Normalized tasks:', len(problems))
print('Dataset:', first.dataset_name)
print('Subset:', first.dataset_subset)
print('Task ID:', first.task_id)
print('Source task ID:', first.source_task_id)
print('Parent task ID:', first.parent_task_id)
print('Entry point:', first.entry_point)
print()
print('Specification preview:')
print(first.prompt_text[:1200])

balanced_10pct = select_problems(
    problems,
    percent=0.10,
    sample_mode='stratified',
    seed=42,
    stratify_by='dataset_subset',
)
print()
print('Balanced 10% sample:', len(balanced_10pct))
print({subset: sum(p.dataset_subset == subset for p in balanced_10pct)
       for subset in sorted({p.dataset_subset for p in balanced_10pct})})

## 7. Configure the LLMs

Layer 1 uses the local model. Layers 2, 3, and the baseline use OpenAI with `reasoning.effort="high"`. A token limit of `0` means the project omits the provider output-token ceiling and evaluates the complete returned response.


In [ ]:
LLM_CONFIG = {
    'layer1_provider': 'hf',
    'layer1_model': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    'layer2_provider': 'openai',
    'layer2_model': 'gpt-5-nano',
    'layer3_provider': 'openai',
    'layer3_model': 'gpt-5-mini',
    'baseline_provider': 'openai',
    'baseline_model': 'gpt-5-mini',
    'openai_reasoning_effort': 'high',
    'openai_text_verbosity': 'low',
    'output_token_limit': 0,  # 0 = no project-imposed output-token ceiling
}
LLM_CONFIG


## 8. Verify that the configured OpenAI model IDs are available

In [ ]:
if os.environ.get('OPENAI_API_KEY'):
    from openai import OpenAI
    client = OpenAI()
    available_model_ids = {model.id for model in client.models.list().data}
    for key in ('layer2_model', 'layer3_model', 'baseline_model'):
        model_id = LLM_CONFIG[key]
        print(f'{key}: {model_id} ->', 'AVAILABLE' if model_id in available_model_ids else 'NOT LISTED')
else:
    print('OPENAI_API_KEY is not loaded; model availability check skipped.')

## 9. Run a no-cost integration smoke test

This checks EvoEval normalization, AST mutation generation, batching, statistics, figures, and ZIP creation without making paid API calls.

In [ ]:
SMOKE_RESULTS = Path('/kaggle/working/cluse_evoeval_smoke')
smoke_command = [
    sys.executable, 'run_pipeline.py',
    '--dataset', str(DATASET_PATH),
    '--dataset-type', DATASET_TYPE,
    '--run-name', 'evoeval_smoke',
    '--limit', '1',
    '--sample-mode', 'stratified',
    '--stratify-by', 'dataset_subset',
    '--seed', '42',
    '--results', str(SMOKE_RESULTS),
    '--max-layers', '3',
    '--max-mutants', '5',
    '--max-probes', '3',
    '--mock',
    '--layer1-max-tokens', '0',
    '--layer2-max-tokens', '0',
    '--layer3-max-tokens', '0',
    '--openai-reasoning-effort', 'high',
    '--openai-text-verbosity', 'low',
    '--display-llm-responses', '1',
    '--display-first-problem-only', '1',
    '--display-compact-call-summary', '1',
    '--verbose-artifacts', '0',
    '--generate-statistics', '1',
    '--bootstrap-samples', '500',
    '--figure-pdf', '0',
    '--zip-output',
]
print(' '.join(smoke_command))
subprocess.run(smoke_command, check=True)

## 10. Run the balanced 10% evaluation

`0.10` selects 50 tasks: 10 from each EvoEval subset. The initial mutant clusters are reused in all later layers; only their surviving members are passed forward. If a layer kills every mutant, remaining layers are skipped automatically.


In [ ]:
RUN_10PCT_EXPERIMENT = False
RESULTS_10PCT = Path('/kaggle/working/cluse_evoeval_10pct')

real_command = [
    sys.executable, 'run_pipeline.py',
    '--dataset', str(DATASET_PATH),
    '--dataset-type', DATASET_TYPE,
    '--run-name', 'evoeval_10pct',
    '--percent', '0.10',
    '--sample-mode', 'stratified',
    '--stratify-by', 'dataset_subset',
    '--seed', '42',
    '--results', str(RESULTS_10PCT),
    '--max-layers', '3',
    '--max-mutants', '25',
    '--max-probes', '6',
    '--layer1-provider', LLM_CONFIG['layer1_provider'],
    '--layer1-model', LLM_CONFIG['layer1_model'],
    '--layer1-max-tokens', '0',
    '--layer1-max-attempts', '3',
    '--layer2-provider', LLM_CONFIG['layer2_provider'],
    '--layer2-model', LLM_CONFIG['layer2_model'],
    '--layer2-fallback-models', '',
    '--layer2-max-attempts', '2',
    '--layer2-max-tokens', '0',
    '--layer3-provider', LLM_CONFIG['layer3_provider'],
    '--layer3-model', LLM_CONFIG['layer3_model'],
    '--layer3-fallback-models', '',
    '--layer3-max-attempts', '1',
    '--layer3-max-tokens', '0',
    '--run-baseline',
    '--baseline-provider', LLM_CONFIG['baseline_provider'],
    '--baseline-model', LLM_CONFIG['baseline_model'],
    '--baseline-fallback-models', '',
    '--baseline-max-iterations', '10',
    '--baseline-max-tokens', '0',
    '--baseline-plateau-patience', '1',
    '--baseline-stop-on-plateau', '1',
    '--layer1-plateau-patience', '1', '--layer1-stop-on-plateau', '1',
    '--layer2-plateau-patience', '1', '--layer2-stop-on-plateau', '1',
    '--layer3-plateau-patience', '1', '--layer3-stop-on-plateau', '1',
    '--require-productive-test', '1',
    '--openai-reasoning-effort', LLM_CONFIG['openai_reasoning_effort'],
    '--openai-text-verbosity', LLM_CONFIG['openai_text_verbosity'],
    '--display-llm-responses', '1',
    '--display-first-problem-only', '1',
    '--display-compact-call-summary', '1',
    '--log-full-llm-io', '0',
    '--verbose-artifacts', '0',
    '--generate-statistics', '1',
    '--bootstrap-samples', '5000',
    '--figure-pdf', '1',
    '--zip-output',
]

print(' '.join(real_command))
if RUN_10PCT_EXPERIMENT:
    subprocess.run(real_command, check=True)
else:
    print('10% run not executed. Set RUN_10PCT_EXPERIMENT=True after the smoke test and model check pass.')

## 11. Verify balanced selection and inspect statistical tables

In [ ]:
RESULTS_TO_VIEW = RESULTS_10PCT if RESULTS_10PCT.exists() else SMOKE_RESULTS

selected_path = RESULTS_TO_VIEW / 'selected_tasks.csv'
if selected_path.exists():
    selected_df = pd.read_csv(selected_path)
    print('Selected tasks:', len(selected_df))
    if 'dataset_subset' in selected_df:
        display(selected_df.groupby('dataset_subset').size().rename('tasks').reset_index())

STATS_DIR = RESULTS_TO_VIEW / 'statistics'
for filename in [
    'effectiveness_summary.csv',
    'paired_comparison_summary.csv',
    'evoeval_subset_summary.csv',
    'evoeval_subset_paired_comparison.csv',
    'efficiency_summary.csv',
    'layer_contribution_summary.csv',
    'operator_difficulty_summary.csv',
    'llm_usage_summary.csv',
]:
    path = STATS_DIR / filename
    if path.exists():
        print()
        print(filename)
        display(pd.read_csv(path))

## 12. Display saved figures

In [ ]:
from IPython.display import Image, display

FIGURES_DIR = RESULTS_TO_VIEW / 'figures'
figures = sorted(FIGURES_DIR.glob('*.png'))
print('Saved figures:', len(figures))
for figure in figures:
    print(figure.name)
    display(Image(filename=str(figure)))

## 13. Run the complete 500-task evaluation

Run only after the balanced 10% experiment has been checked. The model assignments, mutation budget, baseline settings, and seed remain unchanged.

In [ ]:
RUN_FULL_EXPERIMENT = False
RESULTS_FULL = Path('/kaggle/working/cluse_evoeval_full')
full_command = real_command.copy()
full_command[full_command.index('--run-name') + 1] = 'evoeval_full'
full_command[full_command.index('--percent') + 1] = '1.0'
full_command[full_command.index('--results') + 1] = str(RESULTS_FULL)

print(' '.join(full_command))
if RUN_FULL_EXPERIMENT:
    subprocess.run(full_command, check=True)
else:
    print('Full run not executed. Set RUN_FULL_EXPERIMENT=True after validating the 10% results.')

## 14. Package an existing results directory

In [ ]:
RESULTS_TO_PACKAGE = RESULTS_TO_VIEW
zip_path = shutil.make_archive(str(RESULTS_TO_PACKAGE), 'zip', root_dir=RESULTS_TO_PACKAGE)
print('ZIP:', zip_path)
print('Size (MB):', round(Path(zip_path).stat().st_size / (1024 ** 2), 2))

## Interpretation checklist

Report macro and micro mutation scores together. For proposed-versus-baseline claims, use the paired confidence interval, exact sign test, effect sizes, and win/tie/loss counts. Because five evolved tasks can share a HumanEval parent, use the parent-clustered confidence interval in `paired_comparison_summary.csv`. Report per-subset effectiveness and token efficiency, marginal kills by layer, paid tokens and cost per killed mutant, runtime breakdown, and the fixed-versus-plateau baseline comparisons. Keep official-test agreement separate from mutation effectiveness.
Confirm from each task metadata that `reclustering_disabled=true`, and report how often Layer 2 or Layer 3 was skipped because an earlier layer killed all mutants.
